In [37]:
from calibration import (
    load_entries,
    load_template_features_fixed_length,
    dtw_cross_costs,
    compute_pr_curve_from_costs,
    find_threshold_best_f1_from_pr,
    plot_pr_curve_from_costs
)
from feature import(
    make_stft_extractor,
    load_features_from_json
)
from pathlib import Path
import matplotlib.pyplot as plt

In [38]:
#editable variables
feat_method = "stft"
n_calib = 10
n_temps = 5

In [39]:
#Path to get calib_templates
DATA_26 = Path(r"C:\Users\HP\Desktop\Skripsie data\26")
RESULTS_26 = DATA_26 / "template_selections"

#Path to get templates
DATA_29 = Path(r"C:\Users\HP\Desktop\Skripsie data\29")
RESULTS_29 = DATA_29 / "template_selections"

bg_file = "Bground_CALIB_26_20230426_075505_background.json"

st_calib_file = "ST_CALIB_26_20230426_075505_single_tone_templates.json"
mt_calib_file = "MT_CALIB_26_20230426_075505_multi_tone_templates.json"
bt_calib_file = "BT_CALIB_26_20230426_075505_burst_tonal_templates.json"

st_temp_file = "20230429_Templates_20230429_102841_single_tone_templates.json"
mt_temp_file = "20230429_Templates_20230429_102841_multi_tone_templates.json"
bt_temp_file = "20230429_Templates_20230429_102841_burst_tonal_templates.json"


In [40]:
feat_frame_lengths = [0.048, 0.064, 0.096, 0.128]
frame_len_results = {}

window_lengths_to_test = [0.1 ,0.2 ,0.3, 0.4, 0.5, 0.6, 0.7] 

call_types = ["st", "mt", "bt"]

#start and end times for background templates
bg_temps = load_entries(RESULTS_26 / bg_file, n_calib)

st_calib_temps = load_entries(RESULTS_26 / st_calib_file, n_calib)
bt_calib_temps = load_entries(RESULTS_26 / mt_calib_file, n_calib)
mt_calib_temps = load_entries(RESULTS_26 / bt_calib_file, n_calib)

In [41]:
for feat_frame_length in feat_frame_lengths:
    #Type of feature extraction method
    if feat_method == "stft":
        extractor = make_stft_extractor(frame_dur = feat_frame_length)  
    window_results = {}

    for window_len in window_lengths_to_test:
        if feat_frame_length < window_len:
            #determine features
            bg_feats = [load_template_features_fixed_length(template_number, extractor, window_len) for template_number in bg_temps]

            st_calib_feats = [load_template_features_fixed_length(template_number, extractor, window_len) for template_number in st_calib_temps]
            mt_calib_feats = [load_template_features_fixed_length(template_number, extractor, window_len) for template_number in mt_calib_temps]
            bt_calib_feats = [load_template_features_fixed_length(template_number, extractor, window_len) for template_number in bt_calib_temps]

            st_temp_feats = load_features_from_json(RESULTS_29 / st_temp_file, extractor, n_temps)
            mt_temp_feats = load_features_from_json(RESULTS_29 / mt_temp_file, extractor, n_temps)
            bt_temp_feats = load_features_from_json(RESULTS_29 / bt_temp_file, extractor, n_temps)

            feats = {
                            "st": (st_calib_feats, st_temp_feats),
                            "mt": (mt_calib_feats, mt_temp_feats),
                            "bt": (bt_calib_feats, bt_temp_feats)
                        }

            window_results[window_len] = {}
            
            for call_type, (calib_feats, temp_feats) in feats.items():
                call_costs = dtw_cross_costs(temp_feats, calib_feats, extractor.metric)
                bg_costs = dtw_cross_costs(temp_feats, bg_feats, extractor.metric)

                precision, recall, pr_thresholds = compute_pr_curve_from_costs(call_costs, bg_costs)
                threshold, f1 = find_threshold_best_f1_from_pr(precision, recall, pr_thresholds)

                window_results[window_len][call_type] = f1 #{len: {'call': f1}}

        else:
            print(f"{feat_frame_length} larger than {window_len}")
    best_window_len = max(window_results, key=lambda w: min(window_results[w].values())) #finds the window length of the max F1 score of the minimum F1 scores for different lengths
    best_score = min(window_results[best_window_len].values())
    frame_len_results[feat_frame_length] = (best_window_len, best_score)
best_frame_len = max(frame_len_results, key=lambda f: (frame_len_results[f][1], f))
best_window_len, best_score = frame_len_results[best_frame_len]
print(f"Chosen Frame Length: {best_frame_len}")
print(f"Chosen Window Length: {best_window_len}")
print(f"Best F1 Score: {best_score}")

0.128 larger than 0.1
Chosen Frame Length: 0.064
Chosen Window Length: 0.5
Best F1 Score: 0.8505747126436781


In [42]:
if n_temps == 5:
    vote_fracs = [0.2, 0.4, 0.6, 0.8, 1.0]
if n_temps == 10:
    vote_fracs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
vote_frac_results = {}

In [43]:
if feat_method == "stft":
    extractor = make_stft_extractor(frame_dur = best_frame_len)

bg_feats = [load_template_features_fixed_length(template_number, extractor, best_window_len) for template_number in bg_temps]

st_calib_feats = [load_template_features_fixed_length(template_number, extractor, best_window_len) for template_number in st_calib_temps]
mt_calib_feats = [load_template_features_fixed_length(template_number, extractor, best_window_len) for template_number in mt_calib_temps]
bt_calib_feats = [load_template_features_fixed_length(template_number, extractor, best_window_len) for template_number in bt_calib_temps]

st_temp_feats = load_features_from_json(RESULTS_29 / st_temp_file, extractor, n_temps)
mt_temp_feats = load_features_from_json(RESULTS_29 / mt_temp_file, extractor, n_temps)
bt_temp_feats = load_features_from_json(RESULTS_29 / bt_temp_file, extractor, n_temps)

for vote_frac in vote_fracs:
    ...